# Example 5.4 / 2: Distribution Parameters and Bounds

Companion notebook for **5.4 Random Data** in the AMPL lab Documentation, section "Distribution Parameters and Bounds". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Distribution Parameters and Bounds

The AMPL book's function table and its section 7.6 describe the second argument of `Normal` as
the variance. The current AMPL release treats it as the standard deviation, which a large sample
shows directly:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer > 0;                      # Sample size
param mean;                               # Mean of the normal distribution
param sigma > 0;                          # Standard deviation, not variance

param draw {1..n} = Normal(mean, sigma);  # Independent normal draws
param sample_mean = (sum {i in 1..n} draw[i]) / n;  # Average of the draws
# Spread of the draws (divides by n)
param sample_sd = sqrt(sum {i in 1..n} (draw[i] - sample_mean)^2 / n);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.option["randseed"] = 7
ampl.param["n"] = 20000
ampl.param["mean"] = 0
ampl.param["sigma"] = 9

In [5]:
# run.py
print(f"sample standard deviation: {ampl.get_value('sample_sd'):.1f}")

sample standard deviation: 9.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
param n := 20000;
param mean := 0;
param sigma := 9;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option randseed 7;
printf "sample standard deviation: %.1f\n", sample_sd;

sample standard deviation: 9.0
